You are a data generator. Create a synthetic dataset named `sales_ts` with 10,000 rows suitable for time-series analysis.

Schema:
- order_id (string, unique, format: ORD000001 to ORD010000)
- order_date (timestamp, distributed over the last 2 years, realistic daily and seasonal patterns)
- customer_id (string, e.g., CUST001 to CUST500)
- product_id (string, e.g., PROD001 to PROD100)
- category (string, one of: Electronics, Clothing, Grocery, Furniture, Sports)
- region (string, one of: North, South, East, West, Central)
- sales_amount (float, realistic values based on category, range 5 to 5000)
- quantity (integer, range 1 to 10)
- discount (float, range 0 to 0.5, higher discounts during seasonal spikes)
- profit (float, calculated as sales_amount * (0.1 to 0.3) minus discount impact)
- payment_mode (string, one of: Credit Card, Debit Card, UPI, Cash, Net Banking)
- order_status (string, one of: Completed, Cancelled, Returned)

Requirements:
1. Ensure time-series properties:
   - Include daily, weekly, and monthly seasonality
   - Higher sales during weekends and festive periods
   - Slight upward trend over time
2. Maintain realistic relationships:
   - Higher quantity → higher sales_amount
   - Higher discount → slightly lower profit
   - Certain categories (e.g., Electronics) have higher average sales
3. Introduce randomness but keep distributions realistic
4. Ensure no duplicate order_id
5. Output format: CSV with header
6. Ensure data is clean and analysis-ready (no nulls)

Optional (if supported):
- Add holiday/festival spikes (e.g., Diwali, New Year)
- Add slight noise and anomalies for realism

In [0]:
%python
       
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Parameters
n_orders = 10000
n_customers = 500
n_products = 100
categories = ['Electronics', 'Clothing', 'Grocery', 'Furniture', 'Sports']
regions = ['North', 'South', 'East', 'West', 'Central']
payment_modes = ['Credit Card', 'Debit Card', 'UPI', 'Cash', 'Net Banking']
order_statuses = ['Completed', 'Cancelled', 'Returned']
category_sales_mean = {'Electronics': 1200, 'Clothing': 300, 'Grocery': 80, 'Furniture': 700, 'Sports': 400}
category_sales_std = {'Electronics': 800, 'Clothing': 150, 'Grocery': 40, 'Furniture': 400, 'Sports': 200}
festivals = {
    'Diwali': ['2024-11-01', '2025-10-20'],
    'New Year': ['2025-01-01', '2026-01-01'],
    'Christmas': ['2024-12-25', '2025-12-25']
}
festival_days = []
for dates in festivals.values():
    for d in dates:
        festival_days.extend([pd.to_datetime(d) + timedelta(days=i) for i in range(-2, 3)])

# Generate order_id
order_ids = [f"ORD{str(i).zfill(6)}" for i in range(1, n_orders + 1)]

# Generate order_date with seasonality and trend
start_date = datetime.now() - timedelta(days=730)
dates = pd.date_range(start_date, periods=730, freq='D')
date_weights = []
for d in dates:
    # Weekly seasonality: weekends higher
    weekend = d.weekday() >= 5
    # Monthly seasonality: end of month higher
    month_end = d.day > 25
    # Festival spikes
    festival = d in festival_days
    # Upward trend
    trend = (d - start_date).days / 730
    weight = 1 + 0.3 * weekend + 0.2 * month_end + 0.7 * festival + 0.2 * trend
    date_weights.append(weight)
date_probs = np.array(date_weights) / np.sum(date_weights)
order_dates = np.random.choice(dates, size=n_orders, p=date_probs)
order_dates = [pd.Timestamp(d) + timedelta(hours=random.randint(8, 22), minutes=random.randint(0, 59)) for d in order_dates]

# Generate other fields
customer_ids = [f"CUST{str(random.randint(1, n_customers)).zfill(3)}" for _ in range(n_orders)]
product_ids = [f"PROD{str(random.randint(1, n_products)).zfill(3)}" for _ in range(n_orders)]
categories_sample = [random.choices(categories, weights=[0.25,0.25,0.15,0.2,0.15])[0] for _ in range(n_orders)]
regions_sample = [random.choice(regions) for _ in range(n_orders)]
payment_modes_sample = [random.choices(payment_modes, weights=[0.4,0.2,0.2,0.1,0.1])[0] for _ in range(n_orders)]
order_status_sample = [random.choices(order_statuses, weights=[0.92,0.04,0.04])[0] for _ in range(n_orders)]

# Generate quantity, sales_amount, discount, profit
quantity = np.random.randint(1, 11, n_orders)
sales_amount = []
discount = []
profit = []
for i in range(n_orders):
    cat = categories_sample[i]
    base = np.random.normal(category_sales_mean[cat], category_sales_std[cat])
    base = np.clip(base, 5, 5000)
    # Higher sales for higher quantity, with noise
    amt = base * quantity[i] * np.random.uniform(0.9, 1.1)
    # Festival/holiday spike
    if pd.to_datetime(order_dates[i].date()) in festival_days:
        amt *= np.random.uniform(1.2, 1.5)
    # Weekend spike
    if order_dates[i].weekday() >= 5:
        amt *= np.random.uniform(1.05, 1.15)
    amt = np.clip(amt, 5, 5000)
    sales_amount.append(round(amt, 2))
    # Discount: higher during festivals, weekends, and for Clothing/Grocery
    disc_base = 0.05
    if pd.to_datetime(order_dates[i].date()) in festival_days:
        disc_base += 0.25
    if order_dates[i].weekday() >= 5:
        disc_base += 0.1
    if cat in ['Clothing', 'Grocery']:
        disc_base += 0.1
    disc = np.clip(np.random.normal(disc_base, 0.07), 0, 0.5)
    discount.append(round(disc, 3))
    # Profit: sales_amount * margin - discount impact
    margin = np.random.uniform(0.1, 0.3)
    prof = amt * margin - amt * disc * np.random.uniform(0.5, 1.0)
    # Add slight noise/anomalies
    if random.random() < 0.01:
        prof *= np.random.uniform(0.5, 1.5)
    profit.append(round(prof, 2))

# Assemble DataFrame
df = pd.DataFrame({
    'order_id': order_ids,
    'order_date': order_dates,
    'customer_id': customer_ids,
    'product_id': product_ids,
    'category': categories_sample,
    'region': regions_sample,
    'sales_amount': sales_amount,
    'quantity': quantity,
    'discount': discount,
    'profit': profit,
    'payment_mode': payment_modes_sample,
    'order_status': order_status_sample
})

# Ensure no nulls
df = df.fillna(method='ffill').fillna(method='bfill')

# Save as CSV with header
csv_path = '/Volumes/demo/bronze/data/sale_order.csv'
df.to_csv(csv_path, index=False)

# Create Spark DataFrame and store in demo.gold.sales_ts
sale_order_spark = spark.read.csv(csv_path, header=True, inferSchema=True)
sale_order_spark.write.mode('overwrite').saveAsTable('demo.gold.sales_order_ts')